<a href="https://colab.research.google.com/github/yolacin/LocalDispersion/blob/main/ToGitHub_BestKNNC2SearchForMetroData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive
from sklearn.neighbors import NearestNeighbors
from datetime import date, timedelta, datetime

# Mount Google Drive
drive.mount('/content/drive')

# upload the data:
path_to_file = "path_to_file" # path to data.
metro_data = pd.read_csv(path_to_file)

# update column name:
metro_data.columns = ["date_time","traffic_volume"]

# time stamp is of one hour.

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

# ---------------------------------------------------------
# Step 1: Create 168 Hourly Lags (1 Week of History)
# ---------------------------------------------------------
print("Generating lag features...")
# Ensure data is sorted chronologically
metro_data['date_time'] = pd.to_datetime(metro_data['date_time'])
metro_data = metro_data.sort_values('date_time').reset_index(drop=True)

# Generate lags from 1 hour up to 168 hours (1 week)
lag_features = []
for i in range(1, 169):
    col_name = f'lag_{i}'
    metro_data[col_name] = metro_data['traffic_volume'].shift(i)
    lag_features.append(col_name)

# Drop rows containing NaN values generated by lagging
metro_data = metro_data.dropna().reset_index(drop=True)

# Separate features (X) and target (y)
X_all_lags = metro_data[lag_features].values
y_all = metro_data['traffic_volume'].values

In [ ]:


# ---------------------------------------------------------
# Step 2: Chronological Train-Validation-Test Split
# ---------------------------------------------------------
total_rows = len(metro_data)
train_ratio, val_ratio, test_ratio = 0.65, 0.17, 0.18

# Calculate split index positions
train_end = int(total_rows * train_ratio)
val_end = int(total_rows * (train_ratio + val_ratio))

# Split target values
y_train = y_all[:train_end]
y_val = y_all[train_end:val_end]
y_test = y_all[val_end:]  # Saved for later use, untouched for now

# Split lag feature matrices
X_train_lags = X_all_lags[:train_end]
X_val_lags = X_all_lags[train_end:val_end]

print(f"Data successfully split:")
print(f" - Train rows: {len(y_train)}")
print(f" - Validation rows: {len(y_val)}")
print(f" - Test rows: {len(y_test)}")

Data successfully split:
 - Train rows: 13665
 - Validation rows: 3574
 - Test rows: 3785


In [ ]:


# ---------------------------------------------------------
# Step 3: Define Search Parameter Ranges
# ---------------------------------------------------------
w1_range = list(range(2, 25))           # [2, 3, ..., 12]
K1_range = list(range(2, 13))        # [5, 10, ..., 50]

w2_range = list(range(2, 25))          # [12, 13, ..., 40]
K2_range = list(range(2, 13))        # [5, 10, ..., 50]

print(len(w1_range))
print(len(K1_range))
print(len(w2_range))
print(len(K2_range))
# ---------------------------------------------------------
# Step 4: Precompute Validation Predictions
# ---------------------------------------------------------
print("\nPrecomputing short-window model predictions...")
pred_short = {}
for w1 in w1_range:
    # Slice features up to window size w1
    X_tr = X_train_lags[:, :w1]
    X_va = X_val_lags[:, :w1]
    for K1 in K1_range:
        knn = KNeighborsRegressor(n_neighbors=K1, n_jobs=-1)
        knn.fit(X_tr, y_train)
        pred_short[(w1, K1)] = knn.predict(X_va)

print("Precomputing long-window model predictions...")
pred_long = {}
for w2 in w2_range:
    # Slice features up to window size w2
    X_tr = X_train_lags[:, :w2]
    X_va = X_val_lags[:, :w2]
    for K2 in K2_range:
        knn = KNeighborsRegressor(n_neighbors=K2, n_jobs=-1)
        knn.fit(X_tr, y_train)
        pred_long[(w2, K2)] = knn.predict(X_va)

# ---------------------------------------------------------
# Step 5: Evaluate Ensembles to Find the Best Hybrid Pairs
# ---------------------------------------------------------
print("\nEvaluating all 31,900 combinations...")
best_mae = float('inf')
best_params = None

# Grid search by combining the precomputed predictions
for (w1, K1), p_short in pred_short.items():
    for (w2, K2), p_long in pred_long.items():

        # Calculate the ensemble prediction (average of both KNNs)
        ensemble_pred = (p_short + p_long) / 2.0

        # Calculate validation Mean Absolute Error using NumPy for raw speed
        mae = np.mean(np.abs(ensemble_pred - y_val))

        if mae < best_mae:
            best_mae = mae
            best_params = (w1, K1, w2, K2)

# ---------------------------------------------------------
# Step 6: Display the Optimization Results
# ---------------------------------------------------------
opt_w1, opt_K1, opt_w2, opt_K2 = best_params

print("\n" + "="*50)
print("GRID SEARCH COMPLETE")
print("="*50)
print(f"Best Validation MAE: {best_mae:.2f}")
print(f"Optimal Short Window (w1): {opt_w1}")
print(f"Optimal Short K-Neighbors (K1): {opt_K1}")
print(f"Optimal Long Window (w2): {opt_w2}")
print(f"Optimal Long K-Neighbors (K2): {opt_K2}")
print("="*50)

23
11
23
11

Precomputing short-window model predictions...
Precomputing long-window model predictions...

Evaluating all 31,900 combinations...

GRID SEARCH COMPLETE
Best Validation MAE: 199.31
Optimal Short Window (w1): 4
Optimal Short K-Neighbors (K1): 7
Optimal Long Window (w2): 13
Optimal Long K-Neighbors (K2): 5


In [ ]:
np.mean(abs(y_val[:-1]-y_val[1:]))

np.float64(583.6264173031782)

In [ ]:
def MAPE(Y_actual, Y_Predicted):
    Y_Predicted, Y_actual = np.array(Y_Predicted).reshape(-1, 1), np.array(Y_actual).reshape(-1, 1)
    return np.mean(np.abs((Y_actual - Y_Predicted) / Y_actual)) * 100

def MAE(Y_actual, Y_Predicted):
    Y_Predicted, Y_actual = np.array(Y_Predicted).reshape(-1, 1), np.array(Y_actual).reshape(-1, 1)
    return np.mean(np.abs(Y_actual - Y_Predicted))

In [ ]:
MAPE(y_val[:-1],y_val[1:])

np.float64(31.477053751503536)

In [ ]:
MAE(y_val[:-1],y_val[1:])

np.float64(583.6264173031782)